In [ ]:
!pip install qwen_vl_utils
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",device_map="cuda")
from PIL import Image
import torch
# default: Load the model on the available device(s)
# model = Qwen2VLForConditionalGeneration.from_pretrained(
#     # "Qwen/Qwen2-VL-32B-Instruct",
#     "Qwen/Qwen2.5-VL-3B-Instruct",
#     # device_map="cuda",
# )

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen2VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2-VL-7B-Instruct",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384. You can set min_pixels and max_pixels according to your needs, such as a token count range of 256-1280, to balance speed and memory usage.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)


In [ ]:
image_path =  r"/content/game.jpg"
image = Image.open(image_path).convert("RGB") # Ensure the image is in RGB format

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image,
            },
            {"type": "text", "text": "i need your answer to be in this form {the_action_in_the_frame : action type,players : [{team : team color,player_num : num},{team :  team color,player_num : num}] }"},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)


In [ ]:
import os
folder_path = '/content/frames'

# List all files in the folder
files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

print(files)

In [ ]:
for im in files:
  image_path =  '/content/frames/' + im
  image = Image.open(image_path).convert("RGB") # Ensure the image is in RGB format

  messages = [
      {
          "role": "user",
          "content": [
              {
                  "type": "image",
                  "image": image,
              },
              {"type": "text", "text": "i need your answer to be in this form {the_action_in_the_frame : action type,players : [{team : team color,player_num : num},{team :  team color,player_num : num}] }"},
          ],
      }
  ]

  # Preparation for inference
  text = processor.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )
  image_inputs, video_inputs = process_vision_info(messages)
  inputs = processor(
      text=[text],
      images=image_inputs,
      videos=video_inputs,
      padding=True,
      return_tensors="pt",
  )
  inputs = inputs.to("cuda")

  # Inference: Generation of the output
  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_ids_trimmed = [
      out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
  ]
  output_text = processor.batch_decode(
      generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
  )
  print(output_text)


In [ ]:
!pip install pytube
!pip install opencv-python
!pip install yt-dlp

In [ ]:
import yt_dlp

def download_youtube_video(url, save_path="video.mp4"):
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
        'outtmpl': save_path,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    return save_path

# Example usage
video_path = download_youtube_video("https://youtu.be/pBNkcCWbjTk?si=Js1ObZcYiynSR_Ai")
print(f"Downloaded: {video_path}")


In [ ]:
import cv2

video_path = "video.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Cannot open video file.")
else:
    print("Video successfully opened.")

cap.release()


In [ ]:
import cv2

video_path = "video.mp4"
cap = cv2.VideoCapture(video_path)

if cap.isOpened():
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"FPS: {fps}, Total Frames: {total_frames}")
else:
    print("Error: Cannot open video file.")

cap.release()

In [ ]:
!apt-get install -y ffmpeg  # Ensure ffmpeg is installed
!ffmpeg -i video.mp4 -vcodec libx264 -preset ultrafast converted.mp4

In [ ]:
import cv2

def extract_frames(video_path, frame_interval=1):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Cannot open video file.")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video FPS: {fps}, Total Frames: {total_frames}")  # Debugging info

    frame_count = 0
    frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("End of video reached.")
            break

        if frame_count % int(fps * frame_interval) == 0:
            timestamp = frame_count / fps
            frames.append((frame, timestamp))
            print(f"Extracted frame at {timestamp:.2f}s")  # Debugging output

        frame_count += 1

    cap.release()
    print(f"Total frames extracted: {len(frames)}")
    return frames

frames = extract_frames("converted.mp4")


In [ ]:
import os
import cv2
import yt_dlp
import numpy as np

# ------------------------ Step 1: Download YouTube Video ------------------------
def download_youtube_video(url, output_filename="video.mp4"):
    ydl_opts = {
        'format': 'bestvideo+bestaudio/best',
        'outtmpl': output_filename,
        'quiet': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return output_filename

# ------------------------ Step 2: Convert Video for Compatibility ------------------------
def convert_video(input_path, output_path="converted.mp4"):
    os.system(f"ffmpeg -i {input_path} -vcodec libx264 -preset ultrafast {output_path}")
    return output_path

# ------------------------ Step 3: Extract Frames ------------------------
def extract_frames(video_path, output_folder="frames", frame_interval=50):
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"🎥 Video FPS: {fps}, Total Frames: {total_frames}")

    frame_count = 0
    extracted_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break  # End of video

        if frame_count % frame_interval == 0:
            timestamp = frame_count / fps  # Time in seconds
            frame_filename = f"{output_folder}/frame_{frame_count:05d}_t{timestamp:.2f}.jpg"
            cv2.imwrite(frame_filename, frame)
            extracted_count += 1

        frame_count += 1

    cap.release()
    print(f"✅ Total frames extracted: {extracted_count}")

# ------------------------ Step 4: Full Pipeline Execution ------------------------
def process_video(youtube_url):
    print("📥 Downloading video...")
    raw_video = download_youtube_video(youtube_url)

    print("🎬 Converting video format...")
    converted_video = convert_video(raw_video)

    print("🖼️ Extracting frames...")
    extract_frames(converted_video, frame_interval=50)  # Adjust interval as needed

    print("🚀 Processing complete!")


In [ ]:

# ------------------------ Run the Pipeline ------------------------
youtube_url = "https://youtu.be/pBNkcCWbjTk?si=Js1ObZcYiynSR_Ai"  # Replace with actual video URL
process_video(youtube_url)


In [ ]:
!pip install transformers accelerate torch torchvision

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration, BertTokenizerFast
import torch
from PIL import Image
import os
import json

# Load BLIP-2 Model
processor = BlipProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b").to("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
def analyze_frames(frame_folder="frames", output_file="descriptions.txt"):
    frames = sorted(os.listdir(frame_folder))  # Sort frames in order
    results = []

    for frame in frames:
        frame_path = os.path.join(frame_folder, frame)
        image = Image.open(frame_path).convert("RGB")

        # Prepare input for BLIP-2
        inputs = processor(images=image, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

        # Generate caption
        with torch.no_grad():
            caption = model.generate(**inputs)
            description = processor.decode(caption[0], skip_special_tokens=True)

        # Extract timestamp from filename
        timestamp = frame.split("_t")[1].replace(".jpg", "")
        results.append(f"[{timestamp}s] {description}")
        print(f"📝 {timestamp}s → {description}")

    # Save results
    with open(output_file, "w") as f:
        f.write("\n".join(results))

    print(f"✅ Saved descriptions to {output_file}")

# Run the frame analysis
analyze_frames()